In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import json

from src.utils.utils import find_project_root, load_ligand_models

BASE_DIR = find_project_root()

DATA_DIR = BASE_DIR / "data" / "processed" 

from figures.fig_scripts.fig3_functions import *

### Load BMP4 model:

In [10]:
models = load_ligand_models(subset=["BMP4"])
bmp4_model = models["BMP4"]

del models

### Load gene to cluster mappings from figure 2:

In [17]:
#Generated in figure_2.ipynb
cluster_dict_sep = json.load(open(DATA_DIR / "dicts" / "kmeans_5_clusters_dict_sep.json", "r"))

### Generate ligand gene list (up & down not separated) and run GO:

In [18]:
background_genes = bmp4_model.df.columns.to_list()

### Run GO numbered clusters - UPDATED clustering

In [19]:
#print length of each cluster's gene list
for cluster_num, genes in cluster_dict_sep.items():
    print(f"Cluster {cluster_num}: {len(genes)} genes")

Cluster 1_up: 65 genes
Cluster 1_down: 48 genes
Cluster 2_up: 162 genes
Cluster 2_down: 179 genes
Cluster 3_up: 58 genes
Cluster 3_down: 6 genes
Cluster 4_up: 107 genes
Cluster 4_down: 136 genes
Cluster 5_up: 16 genes
Cluster 5_down: 0 genes


In [21]:
go_results_num_clusters_dict = run_go_analysis(
    cluster_dict_sep, background_genes, seperate_up_down=False
)

Processing 1_up...
Processing 1_down...
Processing 2_up...
Processing 2_down...
Processing 3_up...
Processing 3_down...
Processing 4_up...
Processing 4_down...
Processing 5_up...
Processing 5_down...


/home/labs/antebilab/guyilan/master/rec_paper/figures/fig_scripts/fig3_functions.py:36: UserWarning: No genes provided for 5_down, skipping.
  warnings.warn(f"No genes provided for {key}, skipping.")


In [22]:
save_go_results_dataframe(
    go_results_num_clusters_dict,
    DATA_DIR / "go_analysis" / "go_dataframes_sep" ,
)

/home/labs/antebilab/guyilan/master/rec_paper/figures/fig_scripts/fig3_functions.py:96: UserWarning: No GO results for 5_down, skipping.
  warnings.warn(f"No GO results for {group}, skipping.")


In [23]:
go_results_num_clusters_dict = load_go_dataframes_dict(
    path=DATA_DIR / "go_analysis" / "go_dataframes_sep" 
)

In [24]:
go_results_num_clusters_json = generate_json_for_R(go_results_num_clusters_dict)
save_json_for_R(
    go_results_num_clusters_json,
    DATA_DIR / "go_analysis" / "go_jsons" / "kmeans_5_clusters_go_terms_sep.json",
)

In [25]:
env_path = "/home/labs/antebilab/guyilan/.conda/envs/paper_env"

os.environ['PATH'] = f"{env_path}/bin:" + os.environ['PATH']

# Define your custom paths here
my_input = DATA_DIR / "go_analysis" / "go_jsons" / "kmeans_5_clusters_go_terms_sep.json"
my_output = DATA_DIR / "go_analysis" / "go_jsons" / "sim_results_kmeans_5_clusters_sep_from_R.json"
r_script = BASE_DIR / "src" / "analysis" / "get_similarities.R"

# Execute R with the paths as arguments
!Rscript {r_script} {my_input} {my_output}


GOSemSim v2.36.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

Guangchuang Yu. Gene Ontology Semantic Similarity Analysis Using
GOSemSim. In: Kidder B. (eds) Stem Cell Transcriptional Networks.
Methods in Molecular Biology. 2020, 2117:207-215. Humana, New York, NY.

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

Loading required package: AnnotationDbi
Loading required package: stats4
Loading required package: BiocGenerics
Loading required package: generics

Attaching package: ‘generics’

The following objects are masked from ‘package:igraph’:

    components, union

The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union


Attaching package: ‘BiocGenerics’

The following objects are masked from ‘package:igraph’:

    normalize, path

T

numbered_clusters_go_terms.json is processed in src/analysis/get_similarities.R and returend as sim_results_num_clusters_from_R.json

In [26]:
with open(
    DATA_DIR / "go_analysis" / "go_jsons" / "sim_results_kmeans_5_clusters_sep_from_R.json", "r"
) as f:
    go_sim_num_clusters_data = json.load(f)